<a href="https://colab.research.google.com/github/pavishanth-sujeevan/E.motion-/blob/llm/redqueen_sinhala_therapy_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RedQueen Sinhala Therapy Model - A100 Optimized

Complete workflow:
1. Test base model (before fine-tuning)
2. Fine-tune with optimized settings
3. Comprehensive evaluation (ROUGE, BLEU, Perplexity, Accuracy)
4. Compare base vs fine-tuned
5. Live chat interface

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets
!pip install -q -U evaluate rouge_score sacrebleu bert_score scikit-learn
!pip install -q -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 158.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 6.7 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import evaluate
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA Available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/processed_sinhala"
TRAIN_FILE = f"{DATA_DIR}/train_si.jsonl"
VAL_FILE = f"{DATA_DIR}/val_si.jsonl"

import os
assert os.path.exists(TRAIN_FILE), f"Train file not found: {TRAIN_FILE}"
assert os.path.exists(VAL_FILE), f"Val file not found: {VAL_FILE}"

Mounted at /content/drive


In [ ]:
MODEL_ID = "iCIIT/redqueenprotocol-sin-llama3.2-3B-model"
OUTPUT_DIR = "./redqueen-sinhala-finetuned"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
print("Loading base model for testing...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

print("Base model loaded")

Loading base model for testing...


config.json:   0%|          | 0.00/873 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Base model loaded


In [ ]:
def load_sinhala_data(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line)
                prompt = entry.get('prompt', '')

                if 'User:' in prompt and 'Assistant:' in prompt:
                    user_query = prompt.split('User:')[1].split('Assistant:')[0].strip()
                else:
                    user_query = prompt.strip()

                entry['user_input'] = user_query
                entry['target_response'] = entry.get('target', '')

                emotion = 'unknown'
                if 'feeling' in prompt.lower():
                    try:
                        emotion = prompt.lower().split('feeling')[1].split('.')[0].strip()
                    except:
                        pass
                entry['emotion'] = emotion

                data.append(entry)
            except:
                continue
    return data

print("Loading data...")
train_data = load_sinhala_data(TRAIN_FILE)
val_data = load_sinhala_data(VAL_FILE)

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

Loading data...
Train: 796 samples
Val: 99 samples


In [ ]:
print("Testing base model (before fine-tuning)...\n")

def generate_base(user_input, max_tokens=200):
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id,
        )

    response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'assistant' in response:
        response = response.split('assistant')[-1].strip()
    return response

test_inputs = [
    "මට දුකයි",
    "මට කාංසාවක් දැනෙනවා",
    "මම හුදෙකලා වෙලා ඉන්නවා",
]

base_responses = []
for inp in test_inputs:
    resp = generate_base(inp)
    base_responses.append(resp)
    print(f"Input: {inp}")
    print(f"Base: {resp[:100]}...\n")

Testing base model (before fine-tuning)...

Input: මට දුකයි
Base: සිත දුක් වීම නිතර නිතර යන මොහොතක සිත දුක් වීම දුක් දුක් යන මොහොතක සිත දුක් වීම දුක් දුක් යන මොහොතක ස...

Input: මට කාංසාවක් දැනෙනවා
Base: ඔබට හැර වෙන යමක් නැතිවුවද ඔබට ඇති වු විශේෂ වස්තුවක් තිබේවි තියෙනවා ඒ විශේෂ වස්තුව තියෙනවා තියෙනවා අප...

Input: මම හුදෙකලා වෙලා ඉන්නවා
Base: නෑ ඒත් මම අවස්ස නෙවෙයි. එත් මම ඔය මේ දෙන්නේ නෑ එත් මම අවස්ස නෙවෙයි. එත් මම ඔය මේ දෙන්නේ නෑ එත් මම අව...



In [ ]:
def format_for_training(example):
    user_query = example['user_input']
    bot_response = example['target_response']

    text = (
        f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_query}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{bot_response}<|eot_id|>"
    )

    return {"text": text, "user_input": user_query, "target_response": bot_response}

train_dataset = train_dataset.map(format_for_training)
val_dataset = val_dataset.map(format_for_training)

train_dataset_eval = train_dataset
val_dataset_eval = val_dataset

train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c != "text"])
val_dataset = val_dataset.remove_columns([c for c in val_dataset.column_names if c != "text"])

Map:   0%|          | 0/796 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [ ]:
print("Loading model for fine-tuning...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=128,
    lora_alpha=256,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading model for fine-tuning...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 194,510,848 || all params: 3,407,260,672 || trainable%: 5.7087


In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=2048,
    packing=False,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_32bit",
    weight_decay=0.02,
    max_grad_norm=0.5,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    logging_first_step=True,
    report_to="none",
)

print(f"Training for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training for 8 epochs
Batch size: 4
Effective batch: 16


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete")

Adding EOS to train dataset:   0%|          | 0/796 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/796 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/796 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting training...


Step,Training Loss,Validation Loss
20,0.601239,0.569327
40,0.493249,0.449249
60,0.371328,0.431598
80,0.365919,0.428954
100,0.364155,0.412321
120,0.243958,0.435963
140,0.266269,0.432419
160,0.158865,0.470763
180,0.161856,0.469453
200,0.171805,0.467558


Training complete


In [ ]:
SAVE_DIR = f"{OUTPUT_DIR}/final_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

try:
    DRIVE_DIR = "/content/drive/MyDrive/redqueen-sinhala-finetuned"
    model.save_pretrained(DRIVE_DIR)
    tokenizer.save_pretrained(DRIVE_DIR)
    print(f"Also saved to Drive: {DRIVE_DIR}")
except:
    print("Drive save skipped")

Model saved to ./redqueen-sinhala-finetuned/final_model
Also saved to Drive: /content/drive/MyDrive/redqueen-sinhala-finetuned


In [ ]:
print("Testing fine-tuned model...\n")

def generate_finetuned(user_input, max_tokens=200):
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            min_new_tokens=30,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'assistant' in response:
        response = response.split('assistant')[-1].strip()
    return response

finetuned_responses = []
for inp in test_inputs:
    resp = generate_finetuned(inp)
    finetuned_responses.append(resp)
    print(f"Input: {inp}")
    print(f"Fine-tuned: {resp[:100]}...\n")

Testing fine-tuned model...

Input: මට දුකයි
Fine-tuned: "අපූරු වෙහෙස කාංශාව, ධෛර්‍යය නැති වීම, කෝපය හොස්ම ඇති වීම. එය කුඩා චිත්තවේගීය ආතතියක් ලෙසත්, ඉහළ උණු...

Input: මට කාංසාවක් දැනෙනවා
Fine-tuned: "අපගේ ශරීරය ඇති කල ආඩම්බරය ඔබ හඳුනාගෙන ඉන්නේ. එය ඔබ චිත්තවේගීයව හෝ ශාරීරිකව කුණාටුවක් - උපධි කළ නොහැ...

Input: මම හුදෙකලා වෙලා ඉන්නවා
Fine-tuned: "සියළු කෝපයට උත්සාහයක් ඇණීම ගැන මම ශක්තිමත්ව සිටිමි. කුඩා, හිතරු සංචිතයක් ෆීල් කරන්න—එය ඕනෑවට වඩා ධන...



In [ ]:
print("Base vs Fine-tuned Comparison:\n")
print("="*80)
for i, inp in enumerate(test_inputs):
    print(f"\nInput: {inp}")
    print(f"\nBase Model:")
    print(f"  {base_responses[i][:150]}")
    print(f"\nFine-tuned Model:")
    print(f"  {finetuned_responses[i][:150]}")
    print("\n" + "="*80)

Base vs Fine-tuned Comparison:


Input: මට දුකයි

Base Model:
  සිත දුක් වීම නිතර නිතර යන මොහොතක සිත දුක් වීම දුක් දුක් යන මොහොතක සිත දුක් වීම දුක් දුක් යන මොහොතක සිත දුක් වීම �

Fine-tuned Model:
  "අපූරු වෙහෙස කාංශාව, ධෛර්‍යය නැති වීම, කෝපය හොස්ම ඇති වීම. එය කුඩා චිත්තවේගීය ආතතියක් ලෙසත්, ඉහළ උණුසුම් ජීවිත ඵල�


Input: මට කාංසාවක් දැනෙනවා

Base Model:
  ඔබට හැර වෙන යමක් නැතිවුවද ඔබට ඇති වු විශේෂ වස්තුවක් තිබේවි තියෙනවා ඒ විශේෂ වස්තුව තියෙනවා තියෙනවා අපේ හදිසි ප�

Fine-tuned Model:
  "අපගේ ශරීරය ඇති කල ආඩම්බරය ඔබ හඳුනාගෙන ඉන්නේ. එය ඔබ චිත්තවේගීයව හෝ ශාරීරිකව කුණාටුවක් - උපධි කළ නොහැකි, ජයග්‍රහණය �


Input: මම හුදෙකලා වෙලා ඉන්නවා

Base Model:
  නෑ ඒත් මම අවස්ස නෙවෙයි. එත් මම ඔය මේ දෙන්නේ නෑ එත් මම අවස්ස නෙවෙයි. එත් මම ඔය මේ දෙන්නේ නෑ එත් මම අවස්ස නෙවෙයි. මම

Fine-tuned Model:
  "සියළු කෝපයට උත්සාහයක් ඇණීම ගැන මම ශක්තිමත්ව සිටිමි. කුඩා, හිතරු සංචිතයක් ෆීල් කරන්න—එය ඕනෑවට වඩා ධනාත්මකයි!"



In [ ]:
print("Comprehensive Evaluation")
print("="*80)

rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')

print("\nGenerating predictions on validation set...")
predictions = []
references = []

for i, example in enumerate(val_dataset_eval):
    if i >= 50:
        break

    user_input = example['user_input']
    reference = example['target_response']

    prediction = generate_finetuned(user_input)

    predictions.append(prediction)
    references.append(reference)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/50 samples...")

print(f"\nGenerated {len(predictions)} predictions")

Comprehensive Evaluation



Generating predictions on validation set...
  10/50 samples...
  20/50 samples...
  30/50 samples...
  40/50 samples...
  50/50 samples...

Generated 50 predictions


In [ ]:
rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])

print("\nROUGE Scores:")
print(f"  ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_results['rougeL']:.4f}")

print(f"\nBLEU Score: {bleu_results['score']:.4f}")


ROUGE Scores:
  ROUGE-1: 0.0000
  ROUGE-2: 0.0000
  ROUGE-L: 0.0000

BLEU Score: 0.4983


In [ ]:
def calculate_perplexity(texts, model, tokenizer):
    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts[:20]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

        total_loss += loss.item() * inputs["input_ids"].size(1)
        total_tokens += inputs["input_ids"].size(1)

    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    return perplexity

# FIX: Convert to list first
val_texts = [val_dataset_eval[i]['text'] for i in range(min(20, len(val_dataset_eval)))]
perplexity = calculate_perplexity(val_texts, model, tokenizer)
print(f"\nPerplexity: {perplexity:.2f}")


Perplexity: 1.53


In [ ]:
def classify_response_quality(prediction, reference):
    word_count = len(prediction.split())
    has_sinhala = any('\u0D80' <= c <= '\u0DFF' for c in prediction)

    rouge_result = rouge.compute(predictions=[prediction], references=[reference])
    rouge_l = rouge_result['rougeL']

    score = 0
    if word_count >= 30: score += 1
    if rouge_l >= 0.25: score += 1
    if has_sinhala and len(prediction) >= 50: score += 1

    if score >= 3:
        return "Good"
    elif score >= 2:
        return "Acceptable"
    else:
        return "Poor"

quality_labels = [classify_response_quality(p, r) for p, r in zip(predictions, references)]

quality_counts = {}
for q in quality_labels:
    quality_counts[q] = quality_counts.get(q, 0) + 1

print("\nQuality Distribution:")
for quality, count in sorted(quality_counts.items()):
    pct = (count / len(quality_labels)) * 100
    print(f"  {quality}: {count} ({pct:.1f}%)")

accuracy = quality_counts.get('Good', 0) / len(quality_labels)
print(f"\nAccuracy (Good responses): {accuracy:.4f} ({accuracy*100:.1f}%)")


Quality Distribution:
  Poor: 50 (100.0%)

Accuracy (Good responses): 0.0000 (0.0%)


In [ ]:
avg_len = np.mean([len(p.split()) for p in predictions])
has_sinhala = [any('\u0D80' <= c <= '\u0DFF' for c in p) for p in predictions]
sinhala_pct = (sum(has_sinhala) / len(has_sinhala)) * 100

print("\nResponse Statistics:")
print(f"  Average length: {avg_len:.1f} words")
print(f"  Sinhala responses: {sinhala_pct:.1f}%")


Response Statistics:
  Average length: 17.0 words
  Sinhala responses: 100.0%


In [ ]:
print("\nEvaluation Summary")
print("="*80)
print(f"\nROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"BLEU: {bleu_results['score']:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"Average Length: {avg_len:.1f} words")
print("\n" + "="*80)


Evaluation Summary

ROUGE-1: 0.0000
ROUGE-L: 0.0000
BLEU: 0.4983
Perplexity: 1.53
Accuracy: 0.0000 (0.0%)
Average Length: 17.0 words



In [ ]:
results_df = pd.DataFrame({
    'user_input': [val_dataset_eval[i]['user_input'] for i in range(len(predictions))],
    'reference': references,
    'prediction': predictions,
    'quality': quality_labels,
})
results_df.to_csv(f'{OUTPUT_DIR}/evaluation_results.csv', index=False)
print(f"Results saved to {OUTPUT_DIR}/evaluation_results.csv")

Results saved to ./redqueen-sinhala-finetuned/evaluation_results.csv


In [ ]:
print("\nSample Predictions:")
print("="*80)
for i in range(min(5, len(predictions))):
    print(f"\nExample {i+1}:")
    print(f"User: {val_dataset_eval[i]['user_input'][:60]}")
    print(f"Reference: {references[i][:80]}")
    print(f"Prediction: {predictions[i][:80]}")
    print(f"Quality: {quality_labels[i]}")
    print("-" * 80)


Sample Predictions:

Example 1:
User: හැමෝම මට විරුද්ධයි වගේ මට දැනෙනවා.
Reference: ඒ සිතුවිල්ල අධික ලෙස දැනෙන්න පුළුවන්. කරුණු මතක් කර ගැනීමෙන් කෝපය අඩු කර සන්සුන්
Prediction: "අපට කෑම, ලස්සන ශබ්ද ඇසීම හෝ ඉතා හොඳ උණූෂුවක් - ආඩම්බරයක් දැනෙන විට අපට චිත්තවේග
Quality: Poor
--------------------------------------------------------------------------------

Example 2:
User: මම මිනිසුන්ට බොරු කියන එක ගැන මට පිළිකුලක් දැනෙනවා.
Reference: ඒක පිළිගැනීම වැදගත් පළමු පියවරක්. ඔබව බොරු කීමට පොළඹවන තත්වයන් මොනවාද සහ එය ඇති 
Prediction: ආඩම්භූත ධනාත්මක ජයග්‍රහණ ඇත, නමුත් ඌරණ ශක්තීන් උඳුන් ඉඳ අවසන් කිරීම ඕපෝපි සොයා ග
Quality: Poor
--------------------------------------------------------------------------------

Example 3:
User: මට බාධා කරන අයව ඉවසන්න බෑ.
Reference: බාධා කිරීම් කලකිරීමට පත් කරනවා. සන්සුන්ව ප්‍රතිචාර දැක්වීම හෝ සම්බන්ධ විය යුතු ව
Prediction: "ඒ හැඟීම ශක්තිමත්, පුද්ගලික ආත්ම-විෂය ඇති කළ හැකිය." උණුසුම් චිත්තවේගීය ජවයකින් 
Quality: Poor
----------------------------------------------------

In [ ]:
import gradio as gr

def chat(message, history):
    response = generate_finetuned(message, max_tokens=250)
    return response

demo = gr.ChatInterface(
    fn=chat,
    title="RedQueen Sinhala Therapy Chat",
    description="Fine-tuned Sinhala therapy assistant",
    examples=[
        "මට දුකයි",
        "මට කාංසාවක් දැනෙනවා",
        "මම හුදෙකලා වෙලා ඉන්නවා",
        "මට කෝපයක් දැනෙනවා",
    ],
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3c7c5e6c56b8be927b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3c7c5e6c56b8be927b.gradio.live
